In [1]:
import os
import json
from pathlib import Path

import pandas as pd
import serpapi
from dotenv import load_dotenv

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


## 2. Define cities

In [2]:
cities = [
    "Manali",
    "Goa",
    "Jaipur",
    "Udaipur",
    "Rishikesh",
    "Shimla"
]

cities


['Manali', 'Goa', 'Jaipur', 'Udaipur', 'Rishikesh', 'Shimla']

## 3. Create the SerpApi client

In [3]:
load_dotenv()

api_key = os.getenv("SERPAPI_KEY")

if not api_key:
    raise ValueError("SERPAPI_KEY is not found in .env")

client = serpapi.Client(api_key=api_key)

print("SerpApi client ready! ✅")


SerpApi client ready! ✅


## 4. Create a cache

We store every raw response locally.

This lets us experiment with parsing and preprocessing without spending another API search.


In [4]:
cache_dir = Path("../data/raw/multi_city_cache")
cache_dir.mkdir(parents=True, exist_ok=True)

print("Cache directory:", cache_dir)


Cache directory: ..\data\raw\multi_city_cache


## 5. Cached Google Maps search function

In [5]:
def cache_name(city):
    return (
        city.lower()
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
    )


def search_city(city):
    cache_file = (
        cache_dir
        / f"{cache_name(city)}_tourist_attractions.json"
    )

    if cache_file.exists():
        print(f"📦 Cache hit: {city}")
        with open(
            cache_file,
            "r",
            encoding="utf-8"
        ) as file:
            return json.load(file)

    query = f"tourist attractions in {city}"

    print(f"🔎 API search: {query}")

    result = client.search({
        "engine": "google_maps",
        "q": query
    })

    result_dict = dict(result)

    with open(
        cache_file,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            result_dict,
            file,
            indent=4,
            ensure_ascii=False
        )

    print(f"💾 Saved cache: {cache_file.name}")

    return result_dict


## 6. Collect city data

Run this once to collect the six default destinations.

```text
6 cities × 1 query = up to 6 API searches
```


In [6]:
raw_city_results = {}

for city in cities:
    raw_city_results[city] = search_city(city)

print("\n✅ Collection complete")


🔎 API search: tourist attractions in Manali
💾 Saved cache: manali_tourist_attractions.json
🔎 API search: tourist attractions in Goa
💾 Saved cache: goa_tourist_attractions.json
🔎 API search: tourist attractions in Jaipur
💾 Saved cache: jaipur_tourist_attractions.json
🔎 API search: tourist attractions in Udaipur
💾 Saved cache: udaipur_tourist_attractions.json
🔎 API search: tourist attractions in Rishikesh
💾 Saved cache: rishikesh_tourist_attractions.json
🔎 API search: tourist attractions in Shimla
💾 Saved cache: shimla_tourist_attractions.json

✅ Collection complete


## 7. Check result counts

In [7]:
for city, result in raw_city_results.items():
    count = len(
        result.get("local_results", [])
    )

    print(
        f"{city:12} → {count:2} places"
    )


Manali       → 20 places
Goa          → 20 places
Jaipur       → 20 places
Udaipur      → 20 places
Rishikesh    → 20 places
Shimla       → 20 places


## 8. Extract a unified schema

We deliberately keep only fields useful for TravelMate.

The original raw JSON remains untouched in the cache.


In [8]:
def extract_city_places(result, city):
    rows = []

    for place in result.get(
        "local_results",
        []
    ):
        gps = place.get(
            "gps_coordinates",
            {}
        )

        rows.append({
            "place_id":
                place.get("place_id"),

            "name":
                place.get("title"),

            "city":
                city,

            "address":
                place.get("address"),

            "country":
                place.get("country"),

            "latitude":
                gps.get("latitude"),

            "longitude":
                gps.get("longitude"),

            "rating":
                place.get("rating"),

            "reviews":
                place.get("reviews"),

            "category":
                place.get("type")
        })

    return rows


## 9. Build the combined dataset

In [9]:
all_rows = []

for city, result in raw_city_results.items():
    all_rows.extend(
        extract_city_places(
            result,
            city
        )
    )

multi_city_df = pd.DataFrame(
    all_rows
)

print(
    "Combined shape:",
    multi_city_df.shape
)

multi_city_df.head(10)


Combined shape: (120, 10)


,place_id,name,city,address,country,latitude,longitude,rating,reviews,category
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,Manali,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction
1,ChIJDRZ2eeOHBDkRLmyaRD6AZBk,Museum of Himachal Culture & Folk Art,Manali,"Utopia Comlex, Hadimba Temple Rd, near Hadimba...",India,32.246225,77.180570,4.1,1154,Tourist attraction
2,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,Manali,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10844,Tourist attraction
3,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,Manali,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction
4,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,Manali,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction
5,ChIJN0RueliHBDkRI0l6mnxMKk8,Mini Switzerland Manali,Manali,"near Nehru park, Siyal, Manali, Himachal Prade...",India,32.246876,77.189195,4.6,79,Tourist attraction
6,ChIJO_HIPq6JBDkRI2ACToybsFU,Baror Parsha Waterfall,Manali,"Baror, Manali, Himachal Pradesh 175131, India",India,32.206785,77.184900,4.7,489,Tourist attraction
7,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,Manali,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction
8,ChIJlwfAfe6JBDkR4IPtriR7BQQ,Van Vihar National Park,Manali,"65QQ+JJW, Chichoga Rd, Aleo, Manali, Himachal ...",India,32.239113,77.189089,4.2,9049,Tourist attraction
9,ChIJg4mjPAeIBDkR8FyKz0PhtPw,Manali Bazaar,Manali,"65VQ+HQW, Chandigarh-Mandi, Kullu - Naggar - M...",India,32.243991,77.189442,4.3,3992,Tourist attraction


## 10. Remove exact duplicates

In [10]:
before = len(multi_city_df)

multi_city_df = multi_city_df.drop_duplicates(
    subset=["place_id"],
    keep="first"
)

after = len(multi_city_df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)


Rows before: 120
Rows after: 120
Duplicates removed: 0


## 11. Validate data types

In [11]:
numeric_columns = [
    "latitude",
    "longitude",
    "rating",
    "reviews"
]

for col in numeric_columns:
    multi_city_df[col] = pd.to_numeric(
        multi_city_df[col],
        errors="coerce"
    )

text_columns = [
    "place_id",
    "name",
    "city",
    "address",
    "country",
    "category"
]

for col in text_columns:
    multi_city_df[col] = (
        multi_city_df[col]
        .astype("string")
        .str.strip()
    )

multi_city_df.dtypes


place_id      string
name          string
city          string
address       string
country       string
latitude     float64
longitude    float64
rating       float64
reviews        int64
category      string
dtype: object

## 12. Data-quality checks

In [12]:
print("Missing values:")
print(
    multi_city_df.isnull().sum()
)

print(
    "\nDuplicate place IDs:",
    multi_city_df["place_id"].duplicated().sum()
)

print(
    "\nRating range:",
    multi_city_df["rating"].min(),
    "to",
    multi_city_df["rating"].max()
)

print(
    "\nCities represented:"
)

print(
    multi_city_df["city"].value_counts()
)


Missing values:
place_id     0
name         0
city         0
address      0
country      0
latitude     0
longitude    0
rating       0
reviews      0
category     0
dtype: int64

Duplicate place IDs: 0

Rating range: 3.6 to 4.9

Cities represented:
city
Manali       20
Goa          20
Jaipur       20
Udaipur      20
Rishikesh    20
Shimla       20
Name: count, dtype: int64[pyarrow]


## 13. Validate geographic coordinates

In [13]:
invalid_coordinates = multi_city_df[
    (multi_city_df["latitude"] < -90)
    | (multi_city_df["latitude"] > 90)
    | (multi_city_df["longitude"] < -180)
    | (multi_city_df["longitude"] > 180)
]

print(
    "Invalid coordinate rows:",
    len(invalid_coordinates)
)

invalid_coordinates


Invalid coordinate rows: 0


,place_id,name,city,address,country,latitude,longitude,rating,reviews,category


## 14. Add a simple city-aware source query label

This tells us how each record entered our dataset.

Later, we can add richer provenance fields for descriptions, prices and opening hours.


In [14]:
multi_city_df["source_query"] = (
    "tourist attractions in "
    + multi_city_df["city"].astype(str)
)

multi_city_df[
    ["name", "city", "source_query"]
].head(10)


,name,city,source_query
0,Hadimba Devi Temple,Manali,tourist attractions in Manali
1,Museum of Himachal Culture & Folk Art,Manali,tourist attractions in Manali
2,Jogini Falls,Manali,tourist attractions in Manali
3,Old Manali snow point,Manali,tourist attractions in Manali
4,Nehru Kund,Manali,tourist attractions in Manali
5,Mini Switzerland Manali,Manali,tourist attractions in Manali
6,Baror Parsha Waterfall,Manali,tourist attractions in Manali
7,Kullu Manali River rafting,Manali,tourist attractions in Manali
8,Van Vihar National Park,Manali,tourist attractions in Manali
9,Manali Bazaar,Manali,tourist attractions in Manali


## 15. Inspect the city distribution

In [15]:
city_summary = (
    multi_city_df
    .groupby("city")
    .agg(
        places=("place_id", "count"),
        avg_rating=("rating", "mean"),
        total_reviews=("reviews", "sum")
    )
    .reset_index()
    .sort_values(
        "places",
        ascending=False
    )
)

city_summary


,city,places,avg_rating,total_reviews
0,Goa,20,4.325,347287
1,Jaipur,20,4.415,801503
2,Manali,20,4.475,110011
3,Rishikesh,20,4.510,142243
4,Shimla,20,4.430,93205
5,Udaipur,20,4.505,318697


## 16. Inspect the top-rated places in every city

In [16]:
top_by_city = (
    multi_city_df
    .sort_values(
        ["city", "rating", "reviews"],
        ascending=[True, False, False]
    )
    .groupby("city")
    .head(5)
)

top_by_city[[
    "city",
    "name",
    "rating",
    "reviews"
]]


,city,name,rating,reviews
39,Goa,"Shri Nageshi Temple,",4.8,3033
37,Goa,Dudhsagar Trek,4.7,16
21,Goa,Dudhsagar Falls,4.6,32335
30,Goa,Keri Beach,4.6,5070
22,Goa,Sinquerim Fort,4.5,20206
47,Jaipur,Amar Jawan Jyoti,4.7,11722
46,Jaipur,Amber Palace,4.6,172870
58,Jaipur,Central Park,4.6,25012
54,Jaipur,"Sheesh Mahal, Amber Fort",4.6,6301
56,Jaipur,Gaitor Ki Chhatriyan,4.6,6235


## 17. Save the unified raw-style dataset

In [17]:
output_path = (
    "../data/processed/"
    "multi_city_places.csv"
)

multi_city_df.to_csv(
    output_path,
    index=False
)

print(
    f"✅ Unified dataset saved: {output_path}"
)

print(
    "Final shape:",
    multi_city_df.shape
)


✅ Unified dataset saved: ../data/processed/multi_city_places.csv
Final shape: (120, 11)


## 18. Prepare city folders for future enriched data

Each city will eventually have its own processed enrichment while the unified dataset remains the main application source.


In [18]:
processed_dir = Path(
    "../data/processed/cities"
)

for city in cities:
    (processed_dir / cache_name(city)).mkdir(
        parents=True,
        exist_ok=True
    )

print("✅ City processing folders ready")


✅ City processing folders ready
